In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import random
import os
from tqdm import tqdm # Biblioteca para barra de progresso

In [2]:
def plot_beat_segment(segment_df, title="Segmento de Batimento Cardíaco"):
    """Função auxiliar para plotar um segmento de batimento de um DataFrame."""
    if segment_df is None or segment_df.empty:
        print("DataFrame vazio. Nada para plotar.")
        return

    plt.figure(figsize=(12, 6))
    plt.plot(segment_df["sample #"], segment_df["amplitude"], label="Sinal")
    plt.title(title)
    plt.xlabel("Número da Amostra")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

In [3]:
import os
import pandas as pd
import numpy as np

def save_augmented_segment(segment_df, output_dir, file_name):
    """
    Salva um único segmento em CSV, garantindo o formato correto das colunas.

    Parâmetros:
    - segment_df (pd.DataFrame): O DataFrame do segmento a ser salvo.
    - output_dir (str): O diretório onde o arquivo será salvo.
    - file_name (str): O nome do arquivo CSV.
    """
    # Cria uma cópia para evitar modificar o DataFrame original fora da função
    augmented_df = segment_df.copy()
    
    # Renomeia a coluna 'amplitude' de volta para 'channel_0' para o salvamento
    if "amplitude" in augmented_df.columns:
        augmented_df["channel_0"] = augmented_df["amplitude"]
    
    # Garante que as colunas essenciais para o formato de salvamento existam
    if "type" not in augmented_df.columns: 
        augmented_df['type'] = 'unknown' # Adiciona um tipo padrão se não houver
        
    if "sample #" not in augmented_df.columns: 
        augmented_df['sample #'] = np.arange(len(augmented_df)) # Adiciona números de amostra se não houver

    # Seleciona e ordena as colunas para manter um formato consistente
    try:
        augmented_df = augmented_df[["channel_0", "sample #", "type"]]
    except KeyError as e:
        print(f"Erro ao salvar: uma das colunas necessárias não foi encontrada. Colunas disponíveis: {augmented_df.columns}")
        print(f"Erro original: {e}")
        return # Interrompe o salvamento se as colunas estiverem incorretas

    # Constrói o caminho completo do arquivo e salva
    full_path = os.path.join(output_dir, file_name)
    augmented_df.to_csv(full_path, index=False)

In [4]:

def augment_add_sine_pulse(segment_df, amplitude=0.1, frequency=1.0, random_phase=True):
    augmented_df = segment_df.copy()
    y = segment_df["amplitude"].values
    x = np.arange(len(y))
    phase = np.random.uniform(0, 2 * np.pi) if random_phase else 0
    sine_wave = amplitude * np.sin(2 * np.pi * frequency * (x / len(x)) + phase)
    augmented_df["amplitude"] = y + sine_wave
    return augmented_df

In [5]:
# Função para adicionar ruído (Jitter)
def augment_jitter(segment_df, sigma_factor=0.02, variable_sigma=True, seed=None):
    rng = np.random.default_rng(seed)
    augmented_df = segment_df.copy()
    base_sigma = sigma_factor * np.std(segment_df["amplitude"])
    
    if variable_sigma:
        sigma_series = base_sigma * rng.lognormal(mean=0, sigma=0.25, size=len(segment_df))
    else:
        sigma_series = np.full(len(segment_df), base_sigma)
        
    noise = rng.normal(0, sigma_series)
    augmented_df["amplitude"] += noise
    return augmented_df

In [6]:
# --- NOVA FUNÇÃO COMBINADA ---
def augment_sine_and_jitter(segment_df, sine_amplitude=0.1, sine_frequency=1.0, sigma_factor=0.015):
    """
    Aplica uma sequência de aumentos: primeiro SinePulse e depois Jitter.
    """
    # Passo 1: Adicionar a deriva da linha de base
    sine_df = augment_add_sine_pulse(segment_df, amplitude=sine_amplitude, frequency=sine_frequency)
    
    # Passo 2: Adicionar ruído no resultado da primeira transformação
    final_augmented_df = augment_jitter(sine_df, sigma_factor=sigma_factor)
    
    return final_augmented_df

In [7]:

def get_beat_interval_robust_optimized(df, all_peaks, target_rows, nth=0, channel="channel_0"):
    """
    Versão otimizada que recebe um DataFrame e picos pré-calculados.
    """
    # Não precisa mais ler o CSV nem encontrar os picos aqui
    if nth >= len(target_rows):
        print(f"Aviso: nth={nth} está fora do alcance. Anotações encontradas: {len(target_rows)}")
        return None

    center_sample = int(target_rows.iloc[nth]["sample #"])

    # Encontra o índice do pico mais próximo da anotação
    center_peak_index = np.argmin(np.abs(all_peaks - center_sample))
    
    # Verificação de borda
    if center_peak_index == 0 or center_peak_index >= len(all_peaks) - 1:
        return None

    # Pega os picos vizinhos
    start_peak = all_peaks[center_peak_index - 1]
    end_peak = all_peaks[center_peak_index + 1]
    
    # Extrai o segmento
    mask = (df["sample #"] >= start_peak) & (df["sample #"] <= end_peak)
    segment_df = df.loc[mask].copy()

    # Renomeia a coluna para o padrão "amplitude"
    if channel in segment_df.columns:
        segment_df.rename(columns={channel: "amplitude"}, inplace=True)
    
    return segment_df

In [8]:
# --- Execução Otimizada para Arquivo Único (SINE + JITTER) ---

if __name__ == '__main__':
    # --- Parâmetros ---
    csv_file = "mitbih_all_records_renumerada.csv"
    output_file = "augmented_beats_sine_jitter_single_file.csv" # MUDANÇA AQUI
    target_type = "L"
    num_augmentations = 8000
    
    # Parâmetros da detecção de picos
    peak_distance = 180
    peak_height = 0.25
    
    # --- Pré-cálculo ---
    print("1/3 - Carregando arquivo CSV...")
    df_main = pd.read_csv(csv_file)
    
    print(f"2/3 - Encontrando anotações '{target_type}'...")
    target_rows = df_main[df_main["type"] == target_type].reset_index(drop=True)
    
    if len(target_rows) < num_augmentations:
        num_augmentations = len(target_rows)
        print(f"Aviso: Reduzindo para {num_augmentations} aumentos.")

    print("3/3 - Detectando picos R...")
    peaks, _ = find_peaks(df_main["channel_0"].values, distance=peak_distance, height=peak_height)
    
    # --- Loop de Aumento e Coleta ---
    all_augmented_segments = []
    print(f"Gerando {num_augmentations} aumentos de dados com Sine + Jitter...")
    for i in tqdm(range(num_augmentations), desc="Processando Batimentos"):
        segment_df = get_beat_interval_robust_optimized(df_main, peaks, target_rows, nth=i)
        
        if segment_df is not None and not segment_df.empty:
            # MUDANÇA AQUI: Randomiza parâmetros e chama a nova função combinada
            sine_amp = random.uniform(0.05, 0.1)
            sine_freq = random.uniform(0.5, 2.0)
            
            augmented_df = augment_sine_and_jitter(
                segment_df, 
                sine_amplitude=sine_amp, 
                sine_frequency=sine_freq,
                sigma_factor=0.015
            )
            
            # Adiciona um ID para identificar o batimento no arquivo final
            augmented_df['beat_id'] = i
            
            all_augmented_segments.append(augmented_df)
            
    # --- Consolidação e Salvamento Único ---
    if all_augmented_segments:
        print("Consolidando todos os segmentos...")
        final_df = pd.concat(all_augmented_segments, ignore_index=True)
        final_df.rename(columns={'amplitude': 'channel_0'}, inplace=True)
        
        print(f"Salvando em um único arquivo: {output_file}")
        final_df.to_csv(output_file, index=False)
        print("Processo concluído!")
    else:
        print("Nenhum segmento foi gerado.")

1/3 - Carregando arquivo CSV...
2/3 - Encontrando anotações 'L'...
3/3 - Detectando picos R...
Gerando 8000 aumentos de dados com Sine + Jitter...


Processando Batimentos: 100%|███████████████████████████████████████████████████████| 8000/8000 [09:40<00:00, 13.78it/s]


Consolidando todos os segmentos...
Salvando em um único arquivo: augmented_beats_sine_jitter_single_file.csv
Processo concluído!


In [8]:
# --- Execução Otimizada para Múltiplos Arquivos (SINE + JITTER) ---

if __name__ == '__main__':
    # --- Parâmetros ---
    csv_file = "mitbih_all_records_renumerada.csv"
    output_dir = "data_aug_sine_jitter_L_multiple_files/" # MUDANÇA AQUI
    target_type = "R"
    num_augmentations = 7200
    
    # Parâmetros da detecção de picos
    peak_distance = 180
    peak_height = 0.25
    
    # --- Pré-cálculo ---
    os.makedirs(output_dir, exist_ok=True)
    
    print("1/3 - Carregando arquivo CSV...")
    df_main = pd.read_csv(csv_file)
    
    print(f"2/3 - Encontrando anotações '{target_type}'...")
    target_rows = df_main[df_main["type"] == target_type].reset_index(drop=True)
    
    if len(target_rows) < num_augmentations:
        num_augmentations = len(target_rows)
        print(f"Aviso: Reduzindo para {num_augmentations} aumentos.")

    print("3/3 - Detectando picos R...")
    peaks, _ = find_peaks(df_main["channel_0"].values, distance=peak_distance, height=peak_height)
    
    # --- Loop de Aumento e Salvamento ---
    print(f"Gerando e salvando {num_augmentations} arquivos individuais com Sine + Jitter...")
    for i in tqdm(range(num_augmentations), desc="Gerando Arquivos"):
        segment_df = get_beat_interval_robust_optimized(df_main, peaks, target_rows, nth=i)
        
        if segment_df is not None and not segment_df.empty:
            # MUDANÇA AQUI: Randomiza parâmetros e chama a nova função combinada
            sine_amp = random.uniform(0.05, 0.1)
            sine_freq = random.uniform(0.5, 2.0)
            
            augmented_df = augment_sine_and_jitter(
                segment_df, 
                sine_amplitude=sine_amp, 
                sine_frequency=sine_freq,
                sigma_factor=0.015
            )
            
            # MUDANÇA AQUI: Define o novo nome do arquivo
            file_name = f"{target_type}_beat_{i}_aug_sine_jitter.csv"
            save_augmented_segment(augmented_df, output_dir, file_name)
            
    print(f"Processo concluído! {num_augmentations} arquivos salvos em '{output_dir}'.")

1/3 - Carregando arquivo CSV...
2/3 - Encontrando anotações 'R'...
3/3 - Detectando picos R...
Gerando e salvando 7200 arquivos individuais com Sine + Jitter...


Gerando Arquivos: 100%|█████████████████████████████████████████████████████████████| 7200/7200 [09:06<00:00, 13.17it/s]

Processo concluído! 7200 arquivos salvos em 'data_aug_sine_jitter_L_multiple_files/'.
